In [10]:
#importing necessary libraries
import numpy as np 
import pandas as pd
import os
import re
import glob
from pathlib import Path

In [11]:
import sys
print(sys.version)


3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [12]:
df = pd.read_csv("data/raw/patienthistory.csv")

**DATA CLEANING: PATIENT HISTORY CSV FILE**

In [13]:
phist_clean=df.copy()

In [20]:
# Preview the content of the Data file
print(phist_clean.head())
print(phist_clean.shape)
print(phist_clean.columns)
print(phist_clean.tail())
print(len(phist_clean))


   inpatient_number  cerebrovascular_disease  dementia  \
0            857781                        0         0   
1            743087                        0         0   
2            866418                        0         0   
3            775928                        0         0   
4            810128                        0         0   

   chronic_obstructive_pulmonary_disease  connective_tissue_disease  \
0                                      1                          0   
1                                      0                          0   
2                                      0                          0   
3                                      1                          0   
4                                      0                          0   

   peptic_ulcer_disease  diabetes  moderate_to_severe_chronic_kidney_disease  \
0                   0.0         1                                        0.0   
1                   0.0         0                               

**Checking data type of patients id's (column is inpatient_number)**

**Reason: In order to maintain data integrity, consisistency keeping similar numeric ids for data merging at later stages and for Database compatability (Accessing the data)**

In [21]:
# Checking data type of patients id's (column is inpatient_number)
column_dtype = phist_clean['inpatient_number'].dtype
print(f"The column data type is: {column_dtype}")

# Identify non-numeric ids 
non_numeric_mask = pd.to_numeric(phist_clean['inpatient_number'], errors='coerce').isna()
non_numeric_ids = phist_clean[non_numeric_mask]
if non_numeric_ids.empty:
    print("All patient IDs are completely numeric.")
else:
    print(f"Found {len(non_numeric_ids)} row(s) with non-numeric or missing patient IDs.\n")
    print(non_numeric_ids)

The column data type is: int64
All patient IDs are completely numeric.


**To Verify whether DUPLICATE RECORDS are present and print the number of duplicated rows if any**

**Reason: Data file shows the individual patient history for various type of disease. Thers should be unique record for each patient. Any duplication may affect the statistical results** 

In [22]:
#To Verify to find any duplicate records and print the number of duplicated rows if any

duplicate_count = phist_clean.duplicated().sum()
if duplicate_count > 0:
    print(f"Yes, the dataset contains {duplicate_count} duplicate rows.")
else:
    print("No duplicate rows found in the dataset.")


No duplicate rows found in the dataset.


In [18]:
# Another set of verification to find all duplicate records based on both patient ID and drug name in data file
duplicate_patients = phist_clean[phist_clean.duplicated(['inpatient_number'], keep=False)]

if not duplicate_patients.empty:
    # Count how many duplicate rows exist per patient ID
    duplicate_counts_per_patient = duplicate_patients.groupby('inpatient_number').size() // 2
    
    print(f"Found {duplicate_counts_per_patient.count()} unique patient ID(s) with duplicate prescriptions.\n")
    print("--- Patient IDs and their number of duplicate entries ---")
    print(duplicate_counts_per_patient.to_string())
else:
    print("No patient IDs have duplicate entries")

No patient IDs have duplicate entries


**Checking the GENERAL DATA LAYOUT, column types**
**Reason:This is required to generate correct results as below code will help to identify any missing values** 

In [19]:
# Check dataset structure, non-null counts, and data types
phist_clean.info()

# Count the exact number of missing values per column
print(phist_clean.isnull().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 17 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   inpatient_number                           2008 non-null   int64  
 1   cerebrovascular_disease                    2008 non-null   int64  
 2   dementia                                   2008 non-null   int64  
 3   chronic_obstructive_pulmonary_disease      2008 non-null   int64  
 4   connective_tissue_disease                  2008 non-null   int64  
 5   peptic_ulcer_disease                       2006 non-null   float64
 6   diabetes                                   2008 non-null   int64  
 7   moderate_to_severe_chronic_kidney_disease  2006 non-null   float64
 8   hemiplegia                                 2008 non-null   int64  
 9   leukemia                                   2008 non-null   int64  
 10  malignant_lymphoma      

**To change the Column names and then verify the change by displaying the columns list**

**Reason: It should be clear, consistent, and easy to type. Example Column name "patient_id" makes the dataset much easier to work as it's universal term**

In [28]:
phist_clean = phist_clean.rename(columns={'inpatient_number': 'patient_id','type_ii_respiratory_failure': 'type_2_respiratory_failure'})


In [34]:
phist_clean.columns

Index(['patient_id', 'cerebrovascular_disease', 'dementia',
       'chronic_obstructive_pulmonary_disease', 'connective_tissue_disease',
       'peptic_ulcer_disease', 'diabetes',
       'moderate_to_severe_chronic_kidney_disease', 'hemiplegia', 'leukemia',
       'malignant_lymphoma', 'solid_tumor', 'liver_disease', 'aids',
       'cci_score', 'type_2_respiratory_failure', 'acute_renal_failure'],
      dtype='object')

**Verifying the changes made in column names and column values for type_2_respiratory_failure**

In [43]:
phist_clean['type_2_respiratory_failure'] = phist_clean['type_2_respiratory_failure'].replace({
    'nontypeii': 'non-type II',
    'typeii': 'type II'
})

In [44]:
print(phist_clean['type_2_respiratory_failure'].value_counts())

type_2_respiratory_failure
non-type II    1894
type II         114
Name: count, dtype: int64


**Created the new column based on the cleaned text labels for "type_2_respiratory_failure"**
**Reason:For analysis using machine learning or statistical models, it is ideal to use values as 0 and 1. Also It eliminates human error i.e. typos**

In [45]:
phist_clean['type_2_respiratory_failure_Binary'] = phist_clean['type_2_respiratory_failure'].map({
    'non-type II': 0,
    'type II': 1
})
print(phist_clean[['type_2_respiratory_failure', 'type_2_respiratory_failure_Binary']].head(13))

   type_2_respiratory_failure  type_2_respiratory_failure_Binary
0                 non-type II                                  0
1                 non-type II                                  0
2                 non-type II                                  0
3                 non-type II                                  0
4                 non-type II                                  0
5                 non-type II                                  0
6                 non-type II                                  0
7                 non-type II                                  0
8                 non-type II                                  0
9                 non-type II                                  0
10                non-type II                                  0
11                non-type II                                  0
12                    type II                                  1


**Handling the MISSING VALUES by using the NaN**
**Reason:Columns with missing values are in float64(due to default NaN values).It can be converted to Nullable Integer type.This allows whole numbers co-exist with NaN values without displaying as decimals**

In [48]:
cols_with_nan = [
    'peptic_ulcer_disease', 
    'moderate_to_severe_chronic_kidney_disease', 
    'liver_disease', 
    'cci_score'
]
for col in cols_with_nan:
    phist_clean[col] = phist_clean[col].astype('Int64')

#Verifying that types updated successfully to 'Int64' and NaNs are preserved
print("Updated Data Types")
print(phist_clean[cols_with_nan].dtypes)
print("\n--- Preserved Missing Value Counts ---")
print(phist_clean[cols_with_nan].isnull().sum())



Updated Data Types
peptic_ulcer_disease                         Int64
moderate_to_severe_chronic_kidney_disease    Int64
liver_disease                                Int64
cci_score                                    Int64
dtype: object

--- Preserved Missing Value Counts ---
peptic_ulcer_disease                         2
moderate_to_severe_chronic_kidney_disease    2
liver_disease                                1
cci_score                                    5
dtype: int64


**Saving the File**

In [49]:
phist_clean.to_csv('data/cleaned/patienthistory_cleaned.csv', index=False)
print("\nCleaned dataset exported successfully as 'patienthistory_cleaned.csv'!")


Cleaned dataset exported successfully as 'patienthistory_cleaned.csv'!
